# TP4 : Implémentation d'un système RAG (Retrieval-Augmented Generation)

## Présentation du TP

Ce travaux pratiques (TP) est dédié à la mise en place et à l'évaluation d'une architecture **RAG** (*Retrieval-Augmented Generation*). L'objectif principal est de pallier les limites de connaissances d'un grand modèle de langage (LLM) en lui permettant d'extraire et de s'appuyer sur des informations issues de documents externes spécifiques avant de générer une réponse.

Au cours de ce TP, nous allons aborder les étapes clés suivantes :
* **Le Document Loading & Splitting** : Chargement et découpage des documents en blocs (*chunks*).
* **L'Indexation & Vectorisation** : Transformation des textes en vecteurs et stockage dans une base de données vectorielle.
* **Le Retrieval (Recherche)** : Extraction des segments de texte les plus pertinents vis-à-vis d'une requête utilisateur.
* **La Génération & Évaluation** : Injection du contexte dans le prompt du LLM et analyse de la fidélité de ses réponses.


# 0. Preparation de l'environement 

Dans le terminal `uv init` , ` uv venv `, ` Set-ExecutionPolicy -ExecutionPolicy RemoteSigned -Scope Process ` et ` .venv\Scripts\activate ` .  ` uv add ipykernel ` et  ` uv add langchain ` puis ` uv add langchain-openai ` et  `uv add langchain-text-splitters ` puis  `uv add langchain-community ` finalement pour importer la clee api open ai ` uv add python-dotenv  `


In [1]:
import sys
print(sys.version)

3.13.16 (tags/v3.13.16:cbc944f, Sep 30 2026, 19:17:50) [MSC v.1944 64 bit (AMD64)]


In [2]:
import json
import tiktoken
#import pandas as pd||
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFDirectoryLoader, PyPDFLoader 
from langchain_community.vectorstores import Chroma
from dotenv.ipython import load_dotenv
import os

C:\Users\p15s\AppData\Local\Temp\ipykernel_14616\4814559.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader, PyPDFLoader


##  LLM

In [3]:
load_dotenv(override=True)

True

In [4]:
from langchain_groq import ChatGroq
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)

# 1. Loading the pdf, Chunking

Je vais créer un dossier que je vais appeler `PDFs` et à l'intérieur duquel je vais coller ce fichier PDF `nc346.pdf` (pour vous, choisissez votre CV par exemple, ou un fichier que vous connaissez déjà).


In [5]:
pdf_file="./pdfs/chapitre4_RAG.pdf"

In [6]:
pdf_loader=PyPDFLoader(pdf_file)

Et si vous voulez charger un dossier qui contient plusieurs fichiers, vous utilisez `PyPDFDirectoryLoader(path="./pdfs")`.

On va utiliser le découpage (*split*). Je rappelle donc que l'on utilise `RecursiveCharacterTextSplitter` avec l'encodeur `tiktoken`. Voici le modèle Tiktoken utilisé : la taille de chaque bloc (*chunk*) est de 300 tokens et le chevauchement (*overlap*) est de 20.

Maintenant, je vais charger le modèle `TextSplitter`. Notez qu'ici, il manque une dépendance. Pour effectuer le découpage, le système vous demande d'installer `pypdf`.

Faites donc : `uv add pypdf`.


In [7]:
text_splitter=RecursiveCharacterTextSplitter.from_tiktoken_encoder(encoding_name='o200k_base',chunk_size=300,chunk_overlap=20)

Dans le terminal de projet ajouter ` uv add pypdf `

La méthode `load_and_split` va charger le fichier, puis appliquer cette stratégie de découpage. Chaque bloc (*chunk*) aura une taille de 300 tokens avec un chevauchement (*overlap*) de 20. Si vous souhaitez vérifier le nombre total de blocs générés, vous pouvez utiliser `len(chunks)`.


In [8]:
chunks=pdf_loader.load_and_split(text_splitter)
len(chunks)


48

# 2.vectore Store - ChromaDB et Embedding

 *Importer le model d'embedding* 



In [9]:
from langchain_huggingface import HuggingFaceEmbeddings
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

c:\Users\p15s\Documents\TP_Rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Microsoft Visual C++ Redistributable is not installed, this may lead to the DLL load failure.
It can be downloaded at https://aka.ms/vs/17/release/vc_redist.x64.exe


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3740.57it/s]


In [10]:
vectorstore = Chroma.from_documents(
    chunks, embedding_model,
    collection_name="chapitre4_rag",
    persist_directory="./store"
)

Ajouter un dossier "store" et utiliser la commande ` uv add chromadb ` pour ajouter *chromadb*

 Je vais vectoriser les chunks et je les stock dans la base de données vectorielle ici **vectorstore**

# 3.  **Trouver les chunks liée à la questions (Retriever)**

In [11]:
#crée le Retriever 
retriever=vectorstore.as_retriever(search_type='similarity',search_Kwargs={'k',5})
# k=5. Donc je vais lui demander de me chercher les cinq chunks les plus proches à la question.


On va faire un test. J'ai la question suivante : 
> « Quelle est la capitalisation boursière du Maroc pour 2025 ? »

La méthode `retriever.invoke()` a recherché les blocs (*chunks*). Je vais maintenant les afficher.


In [12]:
#test
retrieved_chunks=retriever.invoke(" QU'il est la capitalisation boursière de Maroc pour 2025 ?")
print(retrieved_chunks)
print("nombre de chunks est :",len(retrieved_chunks))


[Document(metadata={'creator': 'OpenAI Prism', 'source': './pdfs/chapitre4_RAG.pdf', 'page': 4, 'total_pages': 48, 'producer': 'pdfTeX-1.40.28', 'page_label': '5', 'keywords': '', 'title': '', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.28 (TeX Live 2025) kpathsea version 6.4.1', 'trapped': '/False', 'moddate': '2026-10-18T14:49:25+00:00', 'author': '', 'subject': '', 'creationdate': '2026-10-18T14:49:25+00:00'}, page_content='Exemple concret d’hallucination\nExemple\nQuestion :« Quel est le PIB du Maroc en 2024? »\nRéponse du LLM (sans RAG) :\n« Le PIB du Maroc en 2024 est de 142 milliards de dollars. »\nImportant\nCe chiffre peut être inventé ou obsolète.\nImportant\nLe LLM ne sait pas qu’il ne sait pas. Il génère une réponse plausible mais\npotentiellement fausse. C’est le problème fondamental des hallucinations.\nL. D – email SMA IA-S3 FST 5 / 48'), Document(metadata={'creationdate': '2026-10-18T14:49:25+00:00', 'creator': 'OpenAI Prism', 'total_pages': 48, 'pr

Maintenant, pour la partie RAG, on va créer notre prompt système (*system prompt*).


# 4.  *Prompt Design*

## 4.1 Prompt template

In [13]:
prompt_template = """
Answer the following question based only on provided context.
The context is about a course on RAG (Retrieval-Augmented Generation), Master IA, FST Beni Mellal.
The context is delimited by <context> tag
The user question is delimited by <question> tag
If the answer is not found in the context, answer: Je ne sais pas!
<context>
{context}
</context>
<question>
{question}
</question>
"""

## 4.2 *Retrieving the Relevant Documents*

### Passage à la partie *retrieving*

Voici ma démarche : je récupère les documents, puis je les **convertis**. Pour chaque document (`relevant_document_chunks`), je sélectionne uniquement le contenu de la page (`d.page_content`), ce qui me génère une liste de textes. Enfin, nous allons concaténer cette liste de *chunks* de texte en utilisant le séparateur `". "`.


In [14]:
user_input="Quelle valeur de température recommande-t-on pour le RAG ? "

relevant_document_chunks=retriever.invoke(user_input)
context_list=[d.page_content for d in relevant_document_chunks]
context_for_query=". ".join(context_list)


context_for_query

'Paramètres de génération\nParamètre Ce qu’il contrôle Valeur RAG\ntemperature Créativité du LLM. À 0 : réponse toujours iden-\ntique et fidèle. À 1 : réponse variée mais risque\nd’inventer.\n0.0 – 0.3\nmax_tokens Longueur maximale de la réponse générée. 500 – 2000\ntop_p Filtre sur le vocabulaire.top_p=0.9signifie : ne\nconsidérer que les mots dont les probabilités cu-\nmulées atteignent 90%.\n0.9\nImportant\nPour le RAG, utilisez une température basse (0.0–0.3) : on veut que le LLM restitue fidèlement le contenu des\ndocuments, pas qu’il improvise.\nL. D – email SMA IA-S3 FST 45 / 48. Quand utiliser le RAG?\nDonnées privées\nou récentes?\nBesoin de\ntraçabilité?\nBudget pour\nfine-tuning?\nRAG Fine-tuning PromptEngineering\nOui Oui Non Oui Non\nL. D – email SMA IA-S3 FST 10 / 48. RAG vs Fine-tuning vs Prompt Engineering\nCritère Prompt Engineering Fine-tuning RAG\nCoût✓Faible✗Élevé★Modéré\nEffort✓Minimal✗Important★Modéré\nFraîcheur données✗Statique✗Statique✓Dynamique\nPrécision domai

In [15]:
len(relevant_document_chunks)

4

## 4.3 Le prompt

Maintenant, on va injecter dans le prompt le contexte et la question en utilisant template format. Vous voyez donc pour template On a injecté le contexte et la question. Et puis donc le si vous voulez afficher le prompt. Qu'est-ce qu'il contient ? Il contient:

 1. le système message:  ( Answer the following question ............  Je ne sais pas! `` <context> ``).
 2. le contexte:  entre ` <context> ` et `` </context> `` .
 3. question:  entre ` <question> ` et ` </question> ` .




In [16]:
prompt=prompt_template.format(context=context_for_query,question=user_input)
print(prompt)


Answer the following question based only on provided context.
The context is about a course on RAG (Retrieval-Augmented Generation), Master IA, FST Beni Mellal.
The context is delimited by <context> tag
The user question is delimited by <question> tag
If the answer is not found in the context, answer: Je ne sais pas!
<context>
Paramètres de génération
Paramètre Ce qu’il contrôle Valeur RAG
temperature Créativité du LLM. À 0 : réponse toujours iden-
tique et fidèle. À 1 : réponse variée mais risque
d’inventer.
0.0 – 0.3
max_tokens Longueur maximale de la réponse générée. 500 – 2000
top_p Filtre sur le vocabulaire.top_p=0.9signifie : ne
considérer que les mots dont les probabilités cu-
mulées atteignent 90%.
0.9
Important
Pour le RAG, utilisez une température basse (0.0–0.3) : on veut que le LLM restitue fidèlement le contenu des
documents, pas qu’il improvise.
L. D – email SMA IA-S3 FST 45 / 48. Quand utiliser le RAG?
Données privées
ou récentes?
Besoin de
traçabilité?
Budget pour
fine

Donc, ça c'est le prompt.

# 5. *Récupérer la réponse de la part de llm*

 Maintenant on va utiliser le prompt. On va poser cette question à LLM.

In [17]:
import sys
!uv pip install --python "{sys.executable}" langchain-groq

Checked 1 package in 15ms


In [18]:
import os, requests
r = requests.get(
    "https://api.groq.com/openai/v1/models",
    headers={"Authorization": f"Bearer {os.environ['GROQ_API_KEY']}"}
)
for m in sorted(x["id"] for x in r.json()["data"]):
    print(m)

allam-2-7b
canopylabs/orpheus-arabic-saudi
canopylabs/orpheus-v1-english
meta-llama/llama-prompt-guard-2-22m
meta-llama/llama-prompt-guard-2-86m
openai/gpt-oss-120b
openai/gpt-oss-20b
openai/gpt-oss-safeguard-20b
qwen/qwen3.8-27b
whisper-large-v3
whisper-large-v3-turbo


In [19]:
from langchain_groq import ChatGroq
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)
groundness_checker = ChatGroq(model="openai/gpt-oss-120b", temperature=0)

In [20]:
resp=llm.invoke(prompt)
from IPython.display import Markdown
print(display(Markdown(resp.content)))

Pour le RAG, on recommande d’utiliser une température basse, typiquement entre **0.0 et 0.3**.

None


## 6. Regroupement dans une application RAG


### Définition de la fonction RAG

Nous allons définir `RAG` comme notre fonction principale de RAG. Elle prendra en paramètres : `query`, `llm` et `prompt_template`.

**Comment fonctionne cette fonction RAG ?**
1. **Le Retriever** : On lui passe la question (`query`), il recherche les documents pertinents et les convertit en une liste de textes. Chaque texte représente le contenu d'une page (`page_content`).
2. **La Concaténation** : Nous allons concaténer cette liste de textes à l'aide d'un séparateur. Ce texte final devient notre **contexte**.
3. **La Génération** : On injecte ce contexte et la question dans le `prompt_template`. On appelle ensuite la méthode `invoke()` et on retourne le contenu généré.

Voici donc la fonction qui nous intéresse. Pour pouvoir la tester, il vous suffira de faire appel à `RAG()` à chaque fois que vous voudrez lui poser une question.



In [21]:
def RAG(query,llm=llm,prompt_template=prompt_template):
    context_docs=retriever.invoke(query)
    context_list=[d.page_content for d in context_docs]
    context_for_query=". ".join(context_list)   
    prompt=prompt_template.format(context=context_for_query, question=query)
    resp=llm.invoke(prompt)
    return resp.content

### Appel de RAG 

In [22]:
resp=RAG(" qu'ils sont les événements culturels dans le sud-est de Maroc ? ")
print(display(Markdown(resp)))

Je ne sais pas!

None


# 7. Évaluation de la réponse du LLM


## 7.1 Architecture

![Logo](./rag_juge.gif)

In [23]:
user_input=" Quelle valeur de température recommande-t-on pour le RAG ? "

Récupérer le context 

In [24]:
relevant_document_chunks=retriever.invoke(user_input)
context_list=[d.page_content for d in relevant_document_chunks]
context_for_query=". ".join(context_list)

In [25]:
user_message_template="""
###Question
{question}
###Context
{context}
###Answer
{answer}
"""

In [26]:
# Réponse d'un RAG
answer=RAG(" Quelle valeur de température recommande-t-on pour le RAG ? ")
print(display(Markdown(answer)))

Pour le RAG, on recommande d’utiliser une température basse, typiquement entre **0,0 et 0,3**.

None


## 7.2- *Groundness*

Le prompt pour le juge 

In [27]:
groundness_rater_system_message="""
Vous êtes chargé d'évaluer des réponses générées par une IA à des questions posées par des utilisateurs.
On vous présentera une question, le contexte utilisé par le système d'IA pour générer la réponse, ainsi qu'une réponse générée par l'IA à la question.
Dans l'entrée, la question commencera par ###Question, le contexte commencera par ###Context, et la réponse générée par L'IA commencera par ###Answer.
Critères d'évaluation :
La tâche consiste à juger dans quelle mesure la réponse respecte la métrique.

1- La métrique n'est pas respectée du tout
2- La métrique n'est respectée que dans une mesure limitée
3- La métrique est respectée dans une bonne mesure
4- La métrique est respectée en grande partie
5- La métrique est entièrement respectée

Métrique :

La réponse doit être dérivée uniquement des informations présentées dans le contexte.

Instructions:

Écrivez d'abord les étapes nécessaires pour évaluer la réponse selon la métrique.
Donnez une explication étape par étape indiquant si la réponse respecte la métrique, en considérant la question et le contexte comme entrées.
Évaluez ensuite dans quelle mesure la métrique est respectée.
Utilisez les informations précédentes pour noter la réponse selon les critères d'évaluation et attribuer un score.
"""

In [28]:
# Le juge doit être plus puissant que le LLM du RAG
from langchain_groq import ChatGroq
groundness_checker = ChatGroq(model="openai/gpt-oss-120b", temperature=0)

In [29]:
def evaluate(system_message, user_message_template,question, model=groundness_checker):
    retrieved_chunks=retriever.invoke(question)
    context_list=[d.page_content for d in retrieved_chunks]
    context=". ".join(context_list)
    answer=RAG(question) 
    prompt=f"""
     { system_message}\n
     USER:
     {user_message_template.format(question=question, context=context, answer=answer)}
     """
    juge_response=model.invoke(prompt)
    return juge_response.content



In [30]:
#évaluer
resp=evaluate(groundness_rater_system_message, user_message_template, user_input)
print(display(Markdown(resp)))

**Étapes pour évaluer la conformité à la métrique « la réponse doit être dérivée uniquement des informations présentées dans le contexte »**

1. **Identifier la question** – Repérer ce qui est demandé.  
2. **Extraire les informations pertinentes du contexte** – Chercher dans le texte fourni les passages qui répondent directement à la question.  
3. **Comparer la réponse générée avec ces passages** – Vérifier que chaque élément de la réponse provient exactement du contexte, sans ajout, interprétation ou information extérieure.  
4. **Détecter toute information non‑présente ou extrapolée** – Si la réponse contient des données qui ne figurent pas dans le contexte, la métrique n’est pas respectée.  
5. **Attribuer un score** selon le degré de conformité (1 = pas du tout, 5 = entièrement).

---

### Application de ces étapes à l’exemple

1. **Question**  
   « Quelle valeur de température recommande‑t‑on pour le RAG ? »

2. **Informations du contexte**  
   - Dans la section *Paramètres de génération* :  
     - « temperature : créativité du LLM… 0.0 – 0.3 »  
   - Dans la partie *Important* :  
     - « Pour le RAG, utilisez une température basse (0.0–0.3) »

3. **Comparaison avec la réponse**  
   - Réponse de l’IA : « Pour le RAG, on recommande d’utiliser une température basse, typiquement entre **0,0 et 0,3**. »  
   - Cette phrase reprend exactement les deux éléments du contexte : la notion de « température basse » et la fourchette « 0.0–0.3 ». Aucun terme supplémentaire n’est introduit.

4. **Recherche d’informations non‑présentes**  
   - Aucun élément de la réponse ne dépasse ou ne modifie le contenu du contexte.  
   - La formulation est fidèle et ne comporte aucune extrapolation.

5. **Évaluation**  
   - La réponse est entièrement dérivée du texte fourni, sans ajout extérieur.

---

## Score attribué

**5 – La métrique est entièrement respectée.**  

La réponse se limite strictement aux informations du contexte et répond précisément à la question posée.

None


### Échelle de notation pour la fidélité au contexte (Faithfulness)


| Score | Signification | Description |
| :--- | :--- | :--- |
| **5 / 5** | **Excellent (Parfait)** | La réponse utilise **uniquement** les informations du contexte. Aucune hallucination, aucune extrapolation. |
| **4 / 5** | **Bon** | La réponse est correcte et s'appuie sur le contexte, mais la formulation pourrait être légèrement plus précise. |
| **3 / 5** | **Moyen** | La réponse est partiellement vraie, mais elle omet un détail important du contexte ou manque de clarté. |
| **2 / 5** | **Médiocre** | La réponse contient des extrapolations ou des informations non vérifiables avec le contexte fourni. |
| **1 / 5** | **Mauvais (Hallucination)** | La réponse est totalement hors-sujet, fausse, ou invente des données qui ne sont pas du tout dans le contexte. |
